# Experiments - LightGCN Recommendation System

This notebook runs hyperparameter experiments and compares models.

In [ ]:
import sys
sys.path.insert(0, '..')

import yaml
import numpy as np
from scipy import sparse
import torch

from src.data.preprocess import preprocess
from src.data.graph import build_adjacency_matrix, symmetric_normalization
from src.data.dataset import InteractionDataset
from src.data.sampler import NegativeSampler
from src.models.lightgcn import LightGCN
from src.training.trainer import Trainer
from src.baselines.popularity import PopularityRecommender
from src.baselines.mf import MatrixFactorizationRecommender
from src.visualization.plots import *
from src.utils.seed import set_seed

In [ ]:
set_seed(42)

with open('../configs/model.yaml') as f:
    model_cfg = yaml.safe_load(f)
with open('../configs/train.yaml') as f:
    train_cfg = yaml.safe_load(f)
with open('../configs/dataset.yaml') as f:
    dataset_cfg = yaml.safe_load(f)

In [ ]:
from pathlib import Path
from src.data.download import download_movielens_100k

data_dir = download_movielens_100k('../data/raw')
meta = preprocess(data_dir, '../data/processed', seed=42)

train_mat = sparse.load_npz('../data/processed/train_interaction.npz')
import pandas as pd
val_df = pd.read_csv('../data/processed/val.csv')
test_df = pd.read_csv('../data/processed/test.csv')

num_users = meta['num_users']
num_items = meta['num_items']

val_mat = sparse.csr_matrix(
    (np.ones(len(val_df), dtype=np.float32),
     (val_df['user_idx'].values, val_df['item_idx'].values)),
    shape=(num_users, num_items)
)
test_mat = sparse.csr_matrix(
    (np.ones(len(test_df), dtype=np.float32),
     (test_df['user_idx'].values, test_df['item_idx'].values)),
    shape=(num_users, num_items)
)

norm_adj = symmetric_normalization(build_adjacency_matrix(train_mat))

In [ ]:
plot_degree_distribution(train_mat, '../outputs/plots')
plot_user_interaction_histogram(train_mat, '../outputs/plots')

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

In [ ]:
all_results = {}

print('=== Baseline: Popularity ===')
pop_model = PopularityRecommender(train_mat)
pop_metrics = {}
for k in [10, 20]:
    hits = 0
    total = 0
    for u in range(num_users):
        if u not in set(test_df['user_idx']):
            continue
        test_items = set(test_df[test_df['user_idx'] == u]['item_idx'].values)
        if not test_items:
            continue
        recs, _ = pop_model.recommend(u, k, train_mat)
        hits += len(set(recs) & test_items)
        total += len(test_items)
    recall = hits / total if total > 0 else 0
    pop_metrics[f'Recall@{k}'] = recall
    print(f'  Recall@{k}: {recall:.4f}')
all_results['Popularity'] = pop_metrics

In [ ]:
print('\n=== Baseline: Matrix Factorization ===')
mf_model = MatrixFactorizationRecommender(train_mat, embedding_dim=64)
mf_model.fit()
mf_metrics = {}
for k in [10, 20]:
    hits = 0
    total = 0
    for u in range(num_users):
        if u not in set(test_df['user_idx']):
            continue
        test_items = set(test_df[test_df['user_idx'] == u]['item_idx'].values)
        if not test_items:
            continue
        recs, _ = mf_model.recommend(u, k, train_mat)
        hits += len(set(recs) & test_items)
        total += len(test_items)
    recall = hits / total if total > 0 else 0
    mf_metrics[f'Recall@{k}'] = recall
    print(f'  Recall@{k}: {recall:.4f}')
all_results['Matrix Factorization'] = mf_metrics

In [ ]:
print('\n=== LightGCN ===')
model = LightGCN(
    num_users=num_users,
    num_items=num_items,
    embedding_dim=model_cfg['embedding_dim'],
    num_layers=model_cfg['num_layers'],
    norm_adj=norm_adj,
    dropout=model_cfg['dropout'],
)

trainer = Trainer(
    model=model,
    train_mat=train_mat,
    val_mat=val_mat,
    test_mat=test_mat,
    lr=train_cfg['learning_rate'],
    weight_decay=train_cfg['weight_decay'],
    neg_samples=train_cfg['neg_samples'],
    gradient_clip=train_cfg['gradient_clip'],
    top_k=train_cfg['top_k'],
    device=device,
    log_dir='../outputs/logs/lightgcn_exp',
    checkpoint_dir='../outputs/checkpoints',
    seed=42,
)

history = trainer.train(
    epochs=train_cfg['epochs'],
    eval_every=train_cfg['eval_every'],
    patience=train_cfg['early_stopping_patience'],
    warmup_epochs=train_cfg['warmup_epochs'],
)

In [ ]:
user_emb, item_emb = model()
lgbcn_metrics = trainer.evaluator.evaluate_all_k(user_emb, item_emb)
print('\nLightGCN Final Metrics:')
for k, v in lgbcn_metrics.items():
    print(f'  {k}: {v:.4f}')
all_results['LightGCN'] = {k: v for k, v in lgbcn_metrics.items() if 'Recall' in k}

In [ ]:
plot_loss_curve(history, '../outputs/plots')
plot_recall_curve(history, k=10, save_dir='../outputs/plots')

In [ ]:
user_emb_np = user_emb.cpu().numpy()
item_emb_np = item_emb.cpu().numpy()
plot_embedding_tsne(user_emb_np, item_emb_np, '../outputs/plots')

In [ ]:
print('\n=== Model Comparison ===')
for model_name, metrics in all_results.items():
    print(f'\n{model_name}:')
    for metric, value in metrics.items():
        print(f'  {metric}: {value:.4f}')